# Four-Station Control

Control experiment isolating **sample size** from **feature-space coverage** in the
AWS-only ($f = 0$) configuration of the land use regression (see the `lur-model` notebook).

With four stations the standardized predictor matrix has rank three, so at most three
predictors can enter the model regardless of where those stations are sited: the
parsimony of the AWS-only fit is imposed by the sample size, not by the environmental
range the stations span. What the sample size does *not* determine is **which**
predictors are retained, and with which sign.

To separate the two, we repeat the AWS-only selection on random four-station subsets of
the LCS network. These share the same rank ceiling and differ only in siting, so any
difference in the retained predictors is attributable to coverage alone.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from zurich_uhi_lur import regr_utils
from zurich_uhi_lur.regr_utils import SpearmanVIFSelector

sns.set_theme(style="ticks")

In [ ]:
# input data paths
stations_gdf_filepath = "../data/processed/stations.gpkg"
station_features_filepath = "../data/processed/stations-features.csv"
ts_df_filepath = "../data/processed/ts-df.csv"

# output
dst_control_filepath = "../data/processed/lur-nfour-control.csv"

# experiment settings (must match lur-model)
# hour of the day (local time)
target_hour = 23
seed = 0
vif_threshold = 5.0

# control settings
n_stations = 4  # matches the number of AWS
n_draws = 200

## Data loading

In [ ]:
station_features_gdf = regr_utils.get_site_features_gdf(
    station_features_filepath, stations_gdf_filepath
)
# Station 534 is co-located with Zürich-Kaserne (identical features), excluded in
# lur-model and therefore excluded here as well.
station_features_gdf = station_features_gdf[station_features_gdf.index != "534"]
aws_gdf = station_features_gdf[station_features_gdf["source"] == "AWS"]
lcs_gdf = station_features_gdf[station_features_gdf["source"] == "LCS"]

feature_columns = station_features_gdf.columns.drop(["geometry", "source"])
print(f"{len(aws_gdf)} AWS, {len(lcs_gdf)} LCS, {len(feature_columns)} predictors")

In [ ]:
ts_df = (
    pd.read_csv(ts_df_filepath, index_col=["heatwave", "time"], parse_dates=["time"])
    .rename_axis(columns=station_features_gdf.index.name)
    .droplevel("heatwave")
)
# timestamps are in UTC, select the target hour in local time
ts_df = ts_df.loc[
    ts_df.index.get_level_values("time")
    .tz_localize("UTC")
    .tz_convert("Europe/Zurich")
    .hour
    == target_hour
]
ts_df.head()

## Rank ceiling at four stations

`StandardScaler` centers the predictors, so four stations span at most a
three-dimensional subspace. Any fourth predictor is then an exact linear combination of
the others, so its variance inflation factor is infinite and the selector rejects it.
The cap at three predictors therefore follows from the admission test itself rather than
from an explicit limit. The condition numbers below illustrate the same rank deficiency.

In [ ]:
X_aws = StandardScaler().fit_transform(aws_gdf[feature_columns])
print(
    f"centered AWS predictor matrix: shape {X_aws.shape}, rank "
    f"{np.linalg.matrix_rank(X_aws)}"
)

# condition number as predictors are added to an arbitrary (here, leading) subset
for k in range(1, 6):
    print(f"  k={k}: cond={np.linalg.cond(X_aws[:, :k]):.3g}")

## Four-station selection

Each trial replicates the $f = 0$ setup of `lur-model` on a four-station subset: the
target is the temperature anomaly relative to the mean of the stations *in that subset*
(as in the main experiment, where the anomaly is taken over the stations being fitted),
repeated across timesteps, and the pipeline is the same standardization → selection →
regression chain.

Feature selection depends only on the predictor--target correlations and the condition
number, so the retained set is the same for MLR, GAM and RF; MLR is used here to obtain
signed coefficients.

In [ ]:
def make_pipeline(model):
    return Pipeline(
        [
            ("scaler", StandardScaler()),
            ("fsel", SpearmanVIFSelector(vif_threshold=vif_threshold)),
            ("model", model),
        ]
    )


def _ols_importances(pipe, X, selected):
    return dict(zip(selected, pipe.named_steps["model"].coef_))


def fit_subset(station_gdf):
    """Fit the LUR pipeline on a station subset, returning signed coefficients."""
    y_ser = (
        ts_df[station_gdf.index]
        .apply(lambda row: row - row.mean(), axis="columns")
        .stack()
        .droplevel("time")
    )
    X_df = station_gdf.loc[y_ser.index, feature_columns]
    scored = regr_utils.score_pipeline(
        make_pipeline(LinearRegression()),
        X_df,
        y_ser,
        importance_fn=_ols_importances,
    )
    return scored["importances"]


# reference: the actual AWS network ($f = 0$ in lur-model)
aws_importances = fit_subset(aws_gdf)
aws_dominant = max(aws_importances, key=lambda feature: abs(aws_importances[feature]))
pd.Series(aws_importances).sort_values(key=abs, ascending=False).rename(
    "AWS coefficient"
)

In [ ]:
rng = np.random.default_rng(seed)

records = []
for draw in range(n_draws):
    subset_gdf = lcs_gdf.sample(n=n_stations, random_state=rng)
    for feature, coefficient in fit_subset(subset_gdf).items():
        records.append({"draw": draw, "feature": feature, "coefficient": coefficient})

control_df = pd.DataFrame(records)
# the dominant predictor is the retained one with the largest absolute coefficient
dominant_ser = control_df.loc[
    control_df.groupby("draw")["coefficient"].apply(lambda s: s.abs().idxmax()),
    "feature",
].reset_index(drop=True)
control_df.head()

## Results

**Figure: dominant predictor across random four-station LCS subsets**, with the predictor that dominates the AWS-only fit highlighted (shown in the notebook only).

In [ ]:
n_selected = control_df.groupby("draw").size()
print(
    f"predictors retained per draw: {n_selected.value_counts().sort_index().to_dict()}"
)

dominant_counts = dominant_ser.value_counts()
aws_dominant_share = dominant_counts.get(aws_dominant, 0) / n_draws
print(
    f"\nAWS dominant predictor: {aws_dominant}\n"
    f"  dominant in {dominant_counts.get(aws_dominant, 0)}/{n_draws} LCS draws "
    f"({aws_dominant_share:.1%}); a different predictor dominates in "
    f"{1 - aws_dominant_share:.1%} of draws"
)
dominant_counts

In [ ]:
# sign stability of the predictors that the AWS fit retains
for feature, aws_coefficient in aws_importances.items():
    feature_df = control_df[control_df["feature"] == feature]
    n_negative = int((feature_df["coefficient"] < 0).sum())
    print(
        f"{feature:38s} AWS {aws_coefficient:+.3f} | retained in "
        f"{len(feature_df):3d}/{n_draws} draws, negative in {n_negative}"
    )

In [ ]:
figwidth, figheight = plt.rcParams["figure.figsize"]

fig, ax = plt.subplots(figsize=(figwidth, figheight))
sns.countplot(
    y=dominant_ser,
    order=dominant_counts.index,
    color="lightgray",
    edgecolor="black",
    ax=ax,
)
# highlight the predictor that dominates the AWS-only fit
for tick_label, patch in zip(ax.get_yticklabels(), ax.patches):
    if tick_label.get_text() == aws_dominant:
        patch.set_facecolor("tab:red")
        tick_label.set_fontweight("bold")
ax.set_xlabel(f"dominant in $n$ of {n_draws} four-station LCS draws")
ax.set_ylabel("")
ax.set_title(
    "Dominant predictor across four-station subsets\n"
    "(red: dominant in the AWS-only fit)"
)
sns.despine(fig)

## Save results

In [ ]:
control_df.to_csv(dst_control_filepath, index=False)